In [1]:
import numpy as np
from xgboost import XGBRegressor 
from sklearn.metrics import r2_score 
from data_processor import DataReader, DataPrep
from sklearn.model_selection import GridSearchCV
from cv_generator import train_val_split, ExpandingWindowCV
from sklearn.ensemble import RandomForestRegressor
from model_eval import *
import pandas as pd
from scipy import stats
from sklearn.preprocessing import LabelEncoder
from functools import partial
from lightgbm import LGBMRegressor
import warnings
import optuna
from sklearn.ensemble import RandomForestClassifier
import joblib as joblib
from tqdm import tqdm

import pickle

In [2]:
import optuna

# You can use Matplotlib instead of Plotly for visualization by simply replacing `optuna.visualization` with
# `optuna.visualization.matplotlib` in the following examples.
from optuna.visualization import plot_contour
from optuna.visualization import plot_edf
from optuna.visualization import plot_intermediate_values
from optuna.visualization import plot_optimization_history
from optuna.visualization import plot_parallel_coordinate
from optuna.visualization import plot_param_importances
from optuna.visualization import plot_rank
from optuna.visualization import plot_slice
from optuna.visualization import plot_timeline

1. Clean the Dataprocessing

In [3]:
daily = 'daily_df.pkl'
intraday = 'intraday_df.pkl'

# Check if the file exists in the current path

if not (os.path.exists(intraday) and os.path.exists(daily)):
    daily_data_path = r'data/daily_data'
    intraday_data_path = r'data/intraday_data'
    intraday_df = DataReader.read_intraday_data(intraday_data_path)
    daily_df = DataReader.read_daily_data(daily_data_path)
    daily_df.to_pickle('daily_df.pkl')
    intraday_df.to_pickle('intraday_df.pkl')

In [15]:
daily_df = pd.read_pickle('daily_df.pkl')
intraday_df =  pd.read_pickle('intraday_df.pkl')
data_prep = DataPrep(intraday_df, daily_df)
target_df = data_prep.get_target(clip_MAD=True, normalize= True)
X_df = data_prep.get_features()#indicators= {'RSI_14':partial(ta.rsi, length= 14), 'RSI_5':partial(ta.rsi, length= 5)})
input_data = X_df.join(target_df[['y', 'y_actual_clipped']], how = 'right')

In [ ]:
X_df

### Train Val Split

In [16]:
features = ['CumReturnResid',  'Rolling_Return_5d', 'Rolling_Return_10d', 'Rolling_Return_20d','NYSE', 'IntradayRSI']#, 'RSI_14']#,'Stock_Split', 'Dividend', 'Rolling_Return_20d', 'EarlyClose', 'NextHoliday']
clipped_returns = [col for col in input_data.columns if 'clipped' in col and 'Return' in col]
features = [f'Rolling_Return_{i}d_clipped' for i in [5, 10]]  + ['CumReturnResid', 'IntradayRSI', 'NYSE']# 'VolumeChangeNormalize']#, 'NYSE'] + 
input_data.dropna(subset=features, inplace=True)
train_data, val_data = train_val_split(input_data, 0.8)
print(features)
x_train, y_train = train_data[features], train_data['y']
x_val, y_val = val_data[features], val_data['y_actual_clipped']#val_data['y']
train_weights = train_data.MDV_63_sqrt.to_numpy()
val_weights = val_data.MDV_63_sqrt.to_numpy()

['Rolling_Return_5d_clipped', 'Rolling_Return_10d_clipped', 'CumReturnResid', 'IntradayRSI', 'NYSE']


In [23]:
val_data["EST_VOL_preday"].to_numpy()

array([0.01222022, 0.01411508, 0.00597373, ..., 0.01417808, 0.00949447,
       0.01531449])

In [17]:
train_data[clipped_returns + ['y']].corr()['y'].sort_values()

CumReturnResid_clipped       -0.009237
Rolling_Return_1d_clipped    -0.004322
Rolling_Return_3d_clipped    -0.002876
Rolling_Return_2d_clipped    -0.002711
Rolling_Return_8d_clipped    -0.000517
Rolling_Return_18d_clipped   -0.000276
Rolling_Return_4d_clipped    -0.000217
Rolling_Return_20d_clipped   -0.000196
Rolling_Return_19d_clipped    0.000005
Rolling_Return_17d_clipped    0.000357
Rolling_Return_6d_clipped     0.000385
Rolling_Return_5d_clipped     0.000426
Rolling_Return_7d_clipped     0.000532
Rolling_Return_9d_clipped     0.000558
Rolling_Return_11d_clipped    0.000811
Rolling_Return_12d_clipped    0.001158
Rolling_Return_16d_clipped    0.001285
Rolling_Return_15d_clipped    0.001305
Rolling_Return_13d_clipped    0.001382
Rolling_Return_10d_clipped    0.001459
Rolling_Return_14d_clipped    0.001659
y                             1.000000
Name: y, dtype: float64

In [18]:
train_data[features + ['y']].corr()['y'].sort_values()


IntradayRSI                  -0.013055
CumReturnResid               -0.006899
NYSE                         -0.000767
Rolling_Return_5d_clipped     0.000426
Rolling_Return_10d_clipped    0.001459
y                             1.000000
Name: y, dtype: float64

### Base random forest grid search

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score

In [10]:
x_train = x_train.drop(columns=x_train.columns[-1])

In [11]:
x_val

Rolling_Return_5d_clipped  \
Date       Id                                        
2014-01-03 BBG000B9WH86                   2.014182   
           BBG000B9XRY4                  -2.793032   
           BBG000B9ZXB4                   0.017785   
           BBG000BB03M1                  -2.297437   
           BBG000BB07P9                   0.224348   
...                                            ...   
2014-12-30 BBG002S5ZRF9                  -0.110907   
           BBG002W96FT9                   2.519768   
           BBG0039320N9                   1.591363   
           BBG005P7Q881                   0.248078   
           BBG006B6PVN9                   1.653275   

                         Rolling_Return_10d_clipped  CumReturnResid  \
Date       Id                                                         
2014-01-03 BBG000B9WH86                    3.982673        0.007069   
           BBG000B9XRY4                   -2.608150       -0.016269   
           BBG000B9ZXB4                    2.224847        0.006342   
           BBG000BB03M1                   -1.632686       -0.001198   
           BBG000BB07P9                   -0.999690        0.002436   
...                                             ...             ...   
2014-12-30 BBG002S5ZRF9                   -6.172149       -0.000395   
           BBG002W96FT9                   -0.697947        0.006691   
           BBG0039320N9                    2.252671       -0.011520   
           BBG005P7Q881                    1.520061        0.001711   
           BBG006B6PVN9                   -1.492236       -0.013142   

                         IntradayRSI  NYSE  
Date       Id                               
2014-01-03 BBG000B9WH86    56.761613   1.0  
           BBG000B9XRY4    30.863562   0.0  
           BBG000B9ZXB4    59.446125   1.0  
           BBG000BB03M1    55.277212   1.0  
           BBG000BB07P9    42.036149   1.0  
...                              ...   ...  
2014-12-30 BBG002S5ZRF9    42.650588   0.0  
           BBG002W96FT9    65.843882   0.0  
           BBG0039320N9    35.596871   1.0  
           BBG005P7Q881    36.935015   0.0  
           BBG006B6PVN9    45.369420   0.0  

[124624 rows x 5 columns]

In [12]:
x_val = x_val.drop(columns=x_val.columns[-1])

In [13]:
def _cv_index(train_data, overlap_period):
        
        cv = []
        date_index = train_data.index.get_level_values('Date')
        date_index_unique = date_index.drop_duplicates()
        for year in date_index.year.unique()[:-1]:
            last_train_idx = date_index.get_loc(date_index[date_index.year <= year][-1]).stop
            first_test_date = date_index_unique[date_index_unique.get_loc(date_index[last_train_idx]) + overlap_period]
            first_test_idx = date_index.get_loc(first_test_date).start
            last_test_idx = date_index.get_loc(date_index[date_index.year <= year + 1][-1]).stop
            cv.append((np.arange(last_train_idx), np.arange(first_test_idx, last_test_idx)))
            
        return cv 

In [14]:
joblib.dump(study, "dumbstudy.pkl")

NameError: name 'study' is not defined

In [ ]:
study2 = joblib.load("dumbstudy.pkl")
print("Best trial until now:")
print(" Value: ", study2.best_trial.value)
print(" Params: ")
for key, value in study2.best_trial.params.items():
    print(f"    {key}: {value}")

In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate, KFold
from sklearn.ensemble import RandomForestRegressor
import optuna
from functools import partial

def conduct_study(regressor, x_train, y_train, y_train_weights):
    
    def objective(trial):
        n_estimators = trial.suggest_int("n_estimators", 100, 300, step=100)#300 not 200
        max_depth = trial.suggest_int("max_depth", 2, 10, step=2) #2,10 NOT 2,4
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10, step=1)# 10 not 2
        
        params = {
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "min_samples_leaf": min_samples_leaf
        }
        
        model = regressor.set_params(**params)
        
        cv_folds = _cv_index(train_data=x_train, overlap_period=1)
        scores = []
        for train_index, test_index in cv_folds:
            X_train_fold, X_test_fold = x_train.iloc[train_index], x_train.iloc[test_index]
            y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
            y_train_weight, y_test_weight = y_train_weights[train_index], y_train_weights[test_index]
            
            model.fit(X_train_fold, y_train_fold, sample_weight=y_train_weight)
            y_pred = model.predict(X_test_fold)
            score = r2_score(y_test_fold,y_pred, sample_weight= y_test_weight)
            #score = model.score(X_test_fold, y_test_fold)  # Adjust scoring method as needed
            scores.append(score)
        
        mean_cv_score = np.mean(scores)
        return mean_cv_score

    study = optuna.create_study(direction='maximize')  # Or 'minimize', depending on the goal

    study.optimize(objective, n_trials=20, n_jobs=14)
    
    return study

# Prediction and evaluation steps here
study_rf = conduct_study(RandomForestRegressor(random_state= 78), x_train,y_train, train_weights)

In [ ]:
joblib.dump(study_rf, "study_rf.pkl")

### Base xgboost Grid Search

In [ ]:
hyperparameter_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 200, 300],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_weight': [1, 5, 10],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}


In [ ]:
import itertools

combinations = list(itertools.product(*hyperparameter_grid.values()))


In [ ]:
import optuna

class GridSearchSampler(optuna.samplers.BaseSampler):
    def __init__(self, combinations, param_names):
        self.combinations = combinations
        self.param_names = param_names
        self.index = 0  # To keep track of which combination to suggest next

    def sample_relative(self, study, trial, search_space):
        # Return an empty dict as this sampler does not use relative sampling
        return {}

    def sample_independent(self, study, trial, param_name, param_distribution):
        # Map the current combination of parameters to their names
        current_combination = dict(zip(self.param_names, self.combinations[self.index % len(self.combinations)]))
        
        # Return the value for the current parameter name
        return current_combination[param_name]
    
    def infer_relative_search_space(self, study, trial):
        # Since we are controlling the grid search manually, we return an empty dict
        # Or you could define the full search space here as well
        return {}
    
    def after_trial(self, study, trial, state, values):
        # Increment index after each trial to move to the next combination
        self.index += 1


In [ ]:
param_names = list(hyperparameter_grid.keys())
sampler = GridSearchSampler(combinations, param_names)

In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate, KFold
from sklearn.ensemble import RandomForestRegressor
import optuna
from functools import partial

def conduct_study(regressor, x_train, y_train, y_train_weights):
    
    def objective(trial):
        max_depth = trial.suggest_int("max_depth", 2, 6, step=2)
        learning_rate = trial.suggest_categorical("learning_rate", [0.01, 0.1, 0.2])
        n_estimators = trial.suggest_categorical("n_estimators", [100, 200, 300])
        subsample = trial.suggest_categorical("subsample", [0.7, 0.9, 1.0])
        colsample_bytree = trial.suggest_categorical("colsample_bytree", [0.5, 0.7, 1.0])
        min_child_weight = trial.suggest_categorical("min_child_weight", [1, 5, 10])
        reg_alpha = trial.suggest_categorical("reg_alpha", [0, 0.1, 1])
        reg_lambda = trial.suggest_categorical("reg_lambda", [1, 1.5, 2])
        
        xgb_params = {
            "max_depth": max_depth,
            "learning_rate": learning_rate,
            "n_estimators": n_estimators,
            "subsample": subsample,
            "colsample_bytree": colsample_bytree,
            "min_child_weight": min_child_weight,
            "reg_alpha": reg_alpha,
            "reg_lambda": reg_lambda
        }

        
        model = regressor.set_params(**xgb_params)
        
        cv_folds = _cv_index(train_data=x_train, overlap_period=1)
        scores = []
        for train_index, test_index in cv_folds:
            X_train_fold, X_test_fold = x_train.iloc[train_index], x_train.iloc[test_index]
            y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
            y_train_weight, y_test_weight = y_train_weights[train_index], y_train_weights[test_index]
            
            model.fit(X_train_fold, y_train_fold, sample_weight=y_train_weight)
            y_pred = model.predict(X_test_fold)
            score = r2_score(y_test_fold,y_pred, sample_weight= y_test_weight)
            #score = model.score(X_test_fold, y_test_fold)  # Adjust scoring method as needed
            scores.append(score)
        
        mean_cv_score = np.mean(scores)
        return mean_cv_score

    study = optuna.create_study(direction='maximize')  
    study.optimize(objective, n_trials=len(combinations))
    
    return study

# Prediction and evaluation steps here
study_xgboost = conduct_study(XGBRegressor(random_state= 78), x_train,y_train, train_weights)

In [ ]:
def objective(trial, params):
    max_depth = trial.suggest_int("max_depth", 2, 6, step=2)
    learning_rate = trial.suggest_categorical("learning_rate", [0.01, 0.1, 0.2])
    n_estimators = trial.suggest_categorical("n_estimators", [100, 200, 300])
    subsample = trial.suggest_categorical("subsample", [0.7, 0.9, 1.0])
    colsample_bytree = trial.suggest_categorical("colsample_bytree", [0.5, 0.7, 1.0])
    min_child_weight = trial.suggest_categorical("min_child_weight", [1, 5, 10])
    reg_alpha = trial.suggest_categorical("reg_alpha", [0, 0.1, 1])
    reg_lambda = trial.suggest_categorical("reg_lambda", [1, 1.5, 2])
    
    xgb_params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda
    }

    
    model = XGBRegressor(random_state= 78).set_params(**xgb_params)
    
    cv_folds = _cv_index(train_data=x_train, overlap_period=1)
    scores = []
    for train_index, test_index in cv_folds:
        X_train_fold, X_test_fold = x_train.iloc[train_index], x_train.iloc[test_index]
        y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
        y_train_weight, y_test_weight = train_weights[train_index], train_weights[test_index]
        
        model.fit(X_train_fold, y_train_fold, sample_weight=y_train_weight)
        y_pred = model.predict(X_test_fold)
        score = r2_score(y_test_fold,y_pred, sample_weight= y_test_weight)
        #score = model.score(X_test_fold, y_test_fold)  # Adjust scoring method as needed
        scores.append(score)
    
    mean_cv_score = np.mean(scores)
    return mean_cv_score

study_xgboost = optuna.create_study(direction='maximize')

#rewrite with tqdm
for params in tqdm(combinations, desc="Evaluating Combinations"):
    trial = study_xgboost.ask()  # Create a new trial
    
    trial.set_user_attr('max_depth', params[0])
    trial.set_user_attr('learning_rate', params[1])
    trial.set_user_attr('n_estimators', params[2])
    trial.set_user_attr('subsample', params[3])
    trial.set_user_attr('colsample_bytree', params[4])
    trial.set_user_attr('min_child_weight', params[5])
    trial.set_user_attr('reg_alpha', params[6])
    trial.set_user_attr('reg_lambda', params[7])
    
    # Evaluate the objective function with the current set of parameters
    score = objective(trial, params)
    
    # Tell Optuna the result of the current trial
    study_xgboost.tell(trial, score)

joblib.dump(study_xgboost, "study_xgb.pkl")

In [ ]:
joblib.dump(study_xgboost, "study_xgb.pkl")

In [ ]:
study_xgboost = joblib.load("study_xgb.pkl")

In [ ]:
study = study_xgboost

In [ ]:
study_xgboost.best_params

In [ ]:
best_params = study_xgboost.best_params
best_model = XGBRegressor(random_state=78, **best_params)
best_model.fit(x_train, y_train, sample_weight=train_weights)


In [ ]:
best_params = study_xgboost.best_params
best_model = XGBRegressor(random_state=78, **best_params)
best_model.fit(x_train, y_train, sample_weight=train_weights)
y_pred = best_model.predict(x_val)
r2_score(y_val.to_numpy(),y_pred, sample_weight= val_weights)

In [ ]:
plot_optimization_history(study)
plot_intermediate_values(study)
plot_parallel_coordinate(study)
plot_parallel_coordinate(study, params=["max_depth", "n_estimators"])
plot_contour(study)
plot_slice(study)
plot_param_importances(study)

In [ ]:
plot_optimization_history(study_xgboost)

### Base LightGBM Grid Search

In [ ]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(verbosity = -1)

lgbm_param_grid = {
    'max_depth': [2, 4, 6],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 150],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.5, 0.7, 1.0],
    'min_child_samples': [1, 5, 10],
    'lambda_l2': [0.05, 0.1, 0.2, 0.3]
}

# expand_wind_lgbm = ExpandingWindowCV(lgbm, lgbm_param_grid) 
# expand_wind_lgbm.fit(x_train, y_train, train_weights)
# model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), val_data.EST_VOL_preday.to_numpy(), val_weights, expand_wind_lgbm.grid_search)
# model_lgbm.to_pickle('lgbm_iter2')
# model_lgbm = load_model('lgbm_iter1')
# print(model_lgbm.grid_cv.best_params_)
best_params = {'colsample_bytree': 1.0, 'lambda_l1': 0.3, 'learning_rate': 0.01, 'max_depth': 4, 'min_child_samples': 5, 'n_estimators': 100, 'subsample': 0.7}
model_lgbm = ModelEval(LGBMRegressor(verbosity = -1),features, x_train, y_train, train_weights, x_val.to_numpy(), y_val.to_numpy(), \
    val_data.EST_VOL_preday.to_numpy(), val_weights,best_params= best_params)
print(model_lgbm.weighted_r2())
print(model_lgbm.feature_importance())
print(model_lgbm.feature_importance_MDA())

In [ ]:
def objective(trial, params):
    max_depth = trial.suggest_int("max_depth", 2, 6, step=2)
    learning_rate = trial.suggest_categorical("learning_rate", [0.01, 0.1, 0.2])
    n_estimators = trial.suggest_categorical("n_estimators", [100, 150])
    subsample = trial.suggest_categorical("subsample", [0.7, 0.9, 1.0])
    colsample_bytree = trial.suggest_categorical("colsample_bytree", [0.5, 0.7, 1.0])
    min_child_samples = trial.suggest_categorical('min_child_samples',[1, 5, 10])
    lambda_l2 = trial.suggest_categorical('lambda_l2', [0.05, 0.1, 0.2, 0.3])
    
    lgbm_params = {
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "n_estimators": n_estimators,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_samples": min_child_samples,
        "lambda_l2": lambda_l2
    }

    
    model = LGBMRegressor(verbosity=-1).set_params(**lgbm_params)
    
    cv_folds = _cv_index(train_data=x_train, overlap_period=1)
    scores = []
    for train_index, test_index in cv_folds:
        X_train_fold, X_test_fold = x_train.iloc[train_index], x_train.iloc[test_index]
        y_train_fold, y_test_fold = y_train.iloc[train_index], y_train.iloc[test_index]
        y_train_weight, y_test_weight = train_weights[train_index], train_weights[test_index]
        
        model.fit(X_train_fold, y_train_fold, sample_weight=y_train_weight)
        y_pred = model.predict(X_test_fold)
        score = r2_score(y_test_fold,y_pred, sample_weight= y_test_weight)
        #score = model.score(X_test_fold, y_test_fold)  # Adjust scoring method as needed
        scores.append(score)
    
    mean_cv_score = np.mean(scores)
    return mean_cv_score

study_lgbm = optuna.create_study(direction='maximize')

#rewrite with tqdm
for params in tqdm(combinations, desc="Evaluating Combinations"):
    trial = study_lgbm.ask()  # Create a new trial
    
    trial.set_user_attr('max_depth', params[0])
    trial.set_user_attr('learning_rate', params[1])
    trial.set_user_attr('n_estimators', params[2])
    trial.set_user_attr('subsample', params[3])
    trial.set_user_attr('colsample_bytree', params[4])
    trial.set_user_attr('min_child_weight', params[5])
    trial.set_user_attr('reg_alpha', params[6])
    trial.set_user_attr('reg_lambda', params[7])
    
    # Evaluate the objective function with the current set of parameters
    score = objective(trial, params)
    
    # Tell Optuna the result of the current trial
    study_lgbm.tell(trial, score)

joblib.dump(study_lgbm, "study_lgbm.pkl")

In [ ]:
plot_optimization_history(study_lgbm)
plot_intermediate_values(study)
plot_parallel_coordinate(study)
plot_parallel_coordinate(study, params=["max_depth", "n_estimators"])
plot_contour(study)
plot_slice(study)
plot_param_importances(study)

In [ ]:
plot_optimization_history(study_lgbm)

In [ ]:
plot_parallel_coordinate(study_lgbm)

In [ ]:
plot_slice(study_lgbm)

In [ ]:
plot_param_importances(study_lgbm)

In [ ]:
study_lgbm.best_params

In [19]:
best_params = {'max_depth': 2,
 'learning_rate': 0.01,
 'n_estimators': 100,
 'subsample': 0.7,
 'colsample_bytree': 1.0,
 'min_child_samples': 5,
 'lambda_l2': 0.05}

In [30]:
best_model = LGBMRegressor(random_state=78, **best_params)
best_model.fit(x_train, y_train, sample_weight=train_weights)
y_pred = best_model.predict(x_val)
r2_score(y_val.to_numpy(),y_pred * val_data["EST_VOL_preday"].to_numpy(), sample_weight= val_weights)

[LightGBM] [Warning] lambda_l2 is set=0.05, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.05
[LightGBM] [Warning] lambda_l2 is set=0.05, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.05
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003240 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1022
[LightGBM] [Info] Number of data points in the train set: 499563, number of used features: 5
[LightGBM] [Info] Start training from score 0.000124
[LightGBM] [Warning] lambda_l2 is set=0.05, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.05


-3.871780653996737e-05